# Task 10: Synthetic PyTorch training-interface smoke

This notebook demonstrates how the deterministic Task 9 input pipeline connects to PyTorch's forward, loss, backward, gradient-clearing, and optimizer interfaces.

It uses one synthetic batch and one test-only scalar parameter. It does not train a protein model, use a biological target, report model quality, or read the project datasets.

In [1]:
import math

import torch

from protein_lm.synthetic.smoke import (
    create_synthetic_smoke_batch,
    run_synthetic_smoke,
)

## 1. Task 9 produces one padded batch

The dataset tokenizes eight synthetic proteins. The deterministic DataLoader selects them, and the collator pads them to one shared width.

In [2]:
batch = create_synthetic_smoke_batch()

print("partition:", batch.partition)
print("shape:", tuple(batch.token_ids.shape))
print("total positions:", batch.token_ids.numel())
print("non-padding positions:", batch.non_padding_mask.sum().item())
print("residue positions:", batch.residue_mask.sum().item())

partition: synthetic
shape: (8, 8)
total positions: 64
non-padding positions: 49
residue positions: 33


## 2. Inspect one protein row

The token row contains BOS, amino-acid tokens, EOS, and right padding. Only positions marked `True` by `residue_mask` contribute to the synthetic loss.

In [3]:
row = batch.accessions.index("SYNTH_001")

print("accession:", batch.accessions[row])
print("token IDs:", batch.token_ids[row].tolist())
print("non-padding mask:", batch.non_padding_mask[row].tolist())
print("residue mask:", batch.residue_mask[row].tolist())
print("coordinates:", batch.residue_coordinates[row].tolist())

accession: SYNTH_001
token IDs: [1, 4, 5, 6, 2, 0, 0, 0]
non-padding mask: [True, True, True, True, True, False, False, False]
residue mask: [False, True, True, True, False, False, False, False]
coordinates: [0, 1, 2, 3, 0, 0, 0, 0]


## 3. Run the CPU smoke twice

Both calls receive the same completed batch. Each call creates a fresh scalar parameter and a fresh SGD optimizer.

In [4]:
cpu_first = run_synthetic_smoke(batch, device="cpu")
cpu_second = run_synthetic_smoke(batch, device="cpu")

print("CPU repetition 1:", cpu_first)
print("CPU repetition 2:", cpu_second)

measurement_names = (
    "initial_parameter",
    "first_loss",
    "first_gradient",
    "updated_parameter",
    "second_loss",
    "second_gradient",
)
cpu_repetitions_agree = all(
    math.isclose(
        getattr(cpu_first, name),
        getattr(cpu_second, name),
        rel_tol=1e-6,
        abs_tol=1e-7,
    )
    for name in measurement_names
)
print("CPU repetitions agree:", cpu_repetitions_agree)

CPU repetition 1: SyntheticSmokeResult(device='cpu', batch_shape=(8, 8), output_shape=(8, 8), total_positions=64, non_padding_positions=49, residue_positions=33, initial_parameter=0.5, first_loss=8.25, first_gradient=33.0, updated_parameter=0.17000000178813934, gradient_cleared=True, second_loss=0.953700065612793, second_gradient=11.220000267028809)
CPU repetition 2: SyntheticSmokeResult(device='cpu', batch_shape=(8, 8), output_shape=(8, 8), total_positions=64, non_padding_positions=49, residue_positions=33, initial_parameter=0.5, first_loss=8.25, first_gradient=33.0, updated_parameter=0.17000000178813934, gradient_cleared=True, second_loss=0.953700065612793, second_gradient=11.220000267028809)
CPU repetitions agree: True


## 4. Connect the PyTorch result to the hand calculation

There are 33 residue positions. At the initial parameter value of 0.5, the expected first loss is `33 × 0.5² = 8.25`, and the expected first gradient is `33 × 2 × 0.5 = 33`.

In [5]:
expected_first_loss = cpu_first.residue_positions * cpu_first.initial_parameter**2
expected_first_gradient = (
    2 * cpu_first.residue_positions * cpu_first.initial_parameter
)

print("expected first loss:", expected_first_loss)
print("observed first loss:", cpu_first.first_loss)
print("expected first gradient:", expected_first_gradient)
print("observed first gradient:", cpu_first.first_gradient)
print("parameter before SGD:", cpu_first.initial_parameter)
print("parameter after SGD:", cpu_first.updated_parameter)
print("gradient cleared:", cpu_first.gradient_cleared)

expected first loss: 8.25
observed first loss: 8.25
expected first gradient: 33.0
observed first gradient: 33.0
parameter before SGD: 0.5
parameter after SGD: 0.17000000178813934
gradient cleared: True


## 5. Run the same batch on Apple MPS

MPS must complete the same interface checks with finite values. Small floating-point differences from CPU are allowed. There is no silent fallback to CPU.

In [6]:
if torch.backends.mps.is_available():
    mps_result = run_synthetic_smoke(batch, device="mps")
    print(mps_result)
else:
    print("MPS is unavailable. No fallback run was performed.")

SyntheticSmokeResult(device='mps', batch_shape=(8, 8), output_shape=(8, 8), total_positions=64, non_padding_positions=49, residue_positions=33, initial_parameter=0.5, first_loss=8.25, first_gradient=33.0, updated_parameter=0.17000001668930054, gradient_cleared=True, second_loss=0.953700065612793, second_gradient=11.220001220703125)


## What this proves

The synthetic protein batch can reach PyTorch's training interface with the expected shape and device. The residue mask excludes BOS, EOS, and padding from the loss. Backpropagation creates finite gradients, SGD changes the parameter, and the old gradient is cleared before a second backward pass.

This does not prove that a model learned protein biology. Week 2 remains the first model-training task.